# Shine-through loads

The part of a neutral beam that is not ionised in the plasma flies on and heats the wall opposite the port, where machines put armour (a beam dump). `deposit` returns this shine-through power and, for every marker, the share of its stream that is left after the plasma (`Deposition.transmission`). `shine_through_loads` follows each surviving stream on its straight line to its first hit on the wall, a simgeometry surface, and tallies its power on the face it hits: the heat flux of every face (power over area), the angle of incidence, the footprint on a map of the wall and the peak heat flux.

The wall can be an axisymmetric equilibrium's wall polygon revolved about $z$ (a triangle mesh with `n_phi` toroidal segments), a `TriangleMesh` read from CAD, or a `FourierSurface` such as the W7-X-like stand-in. The tests check the power balance to roundoff, the heat flux $P\cos\alpha / A$ of a uniform beam on a plate tilted by $\alpha$, and the second-order convergence of the peak of a Gaussian beam as the plate's mesh is refined.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import plasma_plots as pp
import xarray as xr

import freeofcharge as foc
from freeofcharge.shine_through import shine_through_loads

pp.set_backend("plotly")
pp.set_math("latex")
warnings.filterwarnings("ignore", category=UserWarning, module="desc")

## A tokamak at low density

The case of [Stopping and shine-through](/freeofcharge/examples/stopping-and-shine-through/): a Solov'ev tokamak ($R_0 = 6.2$ m, $a = 1.8$ m, elongation 1.7, 5.3 T) with its wall at $\psi_n = 1.3$, and a 1 MeV, 16.5 MW deuterium beam (width 0.3 m, divergence 5 mrad) in a D plasma with 2 % helium at 10 keV. Two aims: the tutorial's tangential one (tangency radius 5.3 m), and a near-perpendicular one (tangency radius 2 m), whose line crosses the magnetic axis's circle and ends on the inner wall.

Each marker here is a beamlet: its position samples the beam's width, and `divergence=0.005` with `resample=20` spreads its surviving stream over twenty lines in a Gaussian cone of 5 mrad. That is the same distribution as markers that sample the divergence themselves, with twenty times the lines on the wall for one deposition. The cone's lines keep their marker's transmission (their paths in the plasma differ by millimetres). The check below compares the two at $n_{e0} = 10^{19}\,\mathrm{m^{-3}}$.

In [ ]:
equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
ions = {"D": (2, 1, 0.96), "He": (4, 2, 0.02)}
aims = {"tangential": 5.3, "perpendicular": 2.0}  # tangency radius [m]


def beamlets(tangency, markers=2000, divergence=0.0, seed=1):
    origin, direction = foc.tangential_injection(tangency, 12.0)
    return foc.NeutralBeam.gaussian(
        origin,
        direction,
        1.0e6,
        16.5e6,
        width=0.3,
        divergence=divergence,
        markers=markers,
        seed=seed,
    )


low = foc.Plasma.parabolic(1e19, 1e4, ions=ions)
start = time.perf_counter()
cone = beamlets(5.3)
cone_deposition = foc.deposit(cone, equilibrium, low, step=0.03, births_per_marker=0)
runs = {
    f"2000 beamlets x 20, seed {seed}": shine_through_loads(
        cone,
        cone_deposition,
        equilibrium,
        divergence=0.005,
        resample=20,
        seed=seed,
    )
    for seed in (2, 3)
}
sampled = beamlets(5.3, markers=40000, divergence=0.005)
runs["40000 markers"] = shine_through_loads(
    sampled,
    foc.deposit(sampled, equilibrium, low, step=0.03, births_per_marker=0),
    equilibrium,
)
for name, loads in runs.items():
    balance = loads.attrs["wall_power"] / loads.attrs["shine_through_power"] - 1
    print(
        f"{name:>29}: shine-through {loads.attrs['shine_through_power'] / 1e6:.3f} MW, "
        f"on the wall {balance:+.0e} of it; peak {loads.attrs['peak_wall_load'] / 1e6:.1f} MW/m^2 "
        f"on a face, {np.nanmax(loads.wall_load_map) / 1e6:.2f} MW/m^2 on the map"
    )
hottest = runs["40000 markers"]
face = int(hottest.wall_load.argmax())
print(
    f"the hottest face: {hottest.area.values[face] * 1e4:.0f} cm^2, "
    f"{hottest.markers.values[face]} lines"
)
print(f"{time.perf_counter() - start:.1f} s")

The power on the wall equals the shine-through power to roundoff in every run. The two beamlet runs (two seeds of the cone) and the 40 000 markers that sample the divergence agree on the peak of the map to 2 % (8.2 to 8.3 MW/m², on bins of about 20 cm). The peak of a single face is noisier: the hottest face (50 cm², about 270 lines) has a counting noise of about 6 %, and the largest of many noisy faces lies above their mean, so the face peaks range from 10.7 to 11.6 MW/m².

## The footprints

The heat flux on the map of the revolved wall: $s$ is the arc length along the wall's outline from the outboard midplane, counter-clockwise (upwards on the outboard side, so the inner wall is around half the perimeter), $\varphi$ the toroidal angle. Each bin holds the power of its faces over their area; the bins (about 20 cm) are coarser than the faces, which smooths the Monte Carlo noise.

In [ ]:
footprints = {}
for aim, tangency in aims.items():
    beam = beamlets(tangency)
    deposition = foc.deposit(beam, equilibrium, low, step=0.03, births_per_marker=0)
    footprints[aim] = shine_through_loads(
        beam,
        deposition,
        equilibrium,
        divergence=0.005,
        resample=20,
        seed=2,
        map_bins=(120, 256),
    )
    loads = footprints[aim]
    hits = loads.hit_position.values
    radius = np.hypot(hits[:, 0], hits[:, 1])
    mean_alpha = np.average(loads.hit_incidence, weights=loads.hit_power)
    print(
        f"{aim:>13}: shine-through {deposition.shine_through_power / 1e6:.2f} MW, hits at R = {np.percentile(radius, 1):.2f} to {np.percentile(radius, 99):.2f} m, "
        f"mean angle of incidence {np.degrees(mean_alpha):.0f} deg, "
        f"peak {loads.attrs['peak_wall_load'] / 1e6:.1f} MW/m^2 at R = {np.hypot(*loads.attrs['peak_position'][:2]):.2f} m"
    )

In [ ]:
for aim, loads in footprints.items():
    heat = loads.wall_load_map.fillna(0.0) / 1e6
    hot = heat.where(heat > 0.02 * heat.max(), drop=True)
    window = heat.sel(
        s=slice(float(hot.s.min()) - 1.0, float(hot.s.max()) + 1.0),
        phi=slice(float(hot.phi.min()) - 0.1, float(hot.phi.max()) + 0.1),
    )
    window.attrs.update(long_name="heat flux", units="MW/m^2")
    window.plasma.plot.view(x="phi", y="s", title=f"{aim} aim").slice().show()

The tangential beam lands on the outboard wall on the far side ($R \approx 8.2$ m, around the midplane at $s \approx 6$ m), the perpendicular one on the inner wall ($R \approx 3.05$ m, $s \approx 17$ m). Both meet the wall about 40° from its normal, as a horizontal line of tangency radius $R_t$ meets a vertical wall at radius $R$ under $\sin\alpha = R_t / R$ (40° for 5.3/8.2 m, 41° for 2/3.05 m). The inner wall's footprint is stretched toroidally, about 1.5 m long, and its peak is higher.

## Peak heat flux against density

The density scan of the tutorial, for both aims: the shine-through power falls steeply with the density, and the peak heat flux with it.

In [ ]:
densities = np.array([1e19, 2e19, 3e19, 5e19, 7e19, 1e20])
peak = np.zeros((len(aims), len(densities)))
shine = np.zeros_like(peak)
for i, tangency in enumerate(aims.values()):
    beam = beamlets(tangency)
    for j, ne0 in enumerate(densities):
        plasma = foc.Plasma.parabolic(ne0, 1e4, ions=ions)
        deposition = foc.deposit(
            beam, equilibrium, plasma, step=0.03, births_per_marker=0
        )
        loads = shine_through_loads(
            beam, deposition, equilibrium, divergence=0.005, resample=20, seed=2
        )
        total = loads.attrs["wall_power"] + loads.attrs["missed_power"]
        assert abs(total / deposition.shine_through_power - 1) < 1e-12
        peak[i, j] = loads.attrs["peak_wall_load"]
        shine[i, j] = deposition.shine_through_power
scan = xr.Dataset(
    {
        "peak_heat_flux": (
            ("aim", "ne0"),
            peak / 1e6,
            {"long_name": "peak heat flux", "units": "MW/m^2"},
        ),
        "shine_through_power": (
            ("aim", "ne0"),
            shine / 1e6,
            {"long_name": "shine-through power", "units": "MW"},
        ),
    },
    coords={
        "aim": list(aims),
        "ne0": (
            "ne0",
            densities,
            {"long_name": "central electron density", "units": "m^-3"},
        ),
    },
)
print("peak heat flux per shine-through power [1/m^2]:")
print((scan.peak_heat_flux / scan.shine_through_power).round(2).to_pandas())
scan.peak_heat_flux.plasma.plot.profiles(
    x="ne0", over="aim", logy=True, cmap="tab10", title=""
)

Between $n_{e0} = 10^{19}$ and $10^{20}\,\mathrm{m^{-3}}$ the peak falls from 10.7 to 0.01 MW/m² for the tangential aim and from 18.8 to 1.6 MW/m² for the perpendicular one, whose path through the plasma is shorter. The peak per shine-through power stays at 1.44 to 1.46 m⁻² for the perpendicular aim; for the tangential aim it falls from 1.24 to 1.13 m⁻², the footprint flattening as the density rises. A likely cause is that the paths across the tangential beam differ (its rays at a smaller tangency radius cross more plasma), so its transmission varies across the beam more than the perpendicular beam's. Every density uses the same markers and cone, so the Monte Carlo noise largely cancels in these ratios.

## W7-X: the armour footprint per source

DESC's W7-X equilibrium on the cached grid of [W7-X deposition](/freeofcharge/examples/w7x-deposition/), the stand-in injectors of `w7x_injectors` and simgeometry's stand-in wall, the plasma boundary offset by 10 cm (`w7x_like_wall`). **Neither the wall nor the injector geometry is W7-X's**: the real wall has NBI armour tiles where the beams land, and the footprints below show what the code computes on the stand-ins, not W7-X's loads. The OP1.2b sources (NI21, sources 1 and 2, all three energy components) at a low density, $n_{e0} = 2\times10^{19}\,\mathrm{m^{-3}}$ and 2 keV. The wall is a `FourierSurface`: its map is over its own angles $(\theta, \varphi)$, $\theta = 0$ on the outboard side.

In [ ]:
from desc.grid import LinearGrid  # noqa: F401  (DESC only for the cached grid)
from simgeometry.data import w7x_like_wall

from freeofcharge.desc_equilibrium import DESCEquilibrium
from freeofcharge.injectors import w7x_injectors

start = time.perf_counter()
cache = Path(
    os.environ.get("FREEOFCHARGE_CACHE", Path.home() / ".cache" / "freeofcharge")
)
w7x = foc.GriddedEquilibrium.cached(
    cache / "w7x_desc_128_64_128.nc",
    DESCEquilibrium.from_example("W7-X"),
    (128, 64, 128),
)
wall = w7x_like_wall(0.10)
plasma = foc.Plasma.parabolic(
    2e19, 2e3, ions={"H": (1, 1, 1.0)}, radial_coordinate="toroidal"
)
(box,) = w7x_injectors(sources={"NI21": [1, 2]})
armour = {}
for k, source in enumerate(box.sources):
    beam = source.beam(400, seed=10 * k)
    deposition = foc.deposit(beam, w7x, plasma, step=0.01, births_per_marker=0)
    armour[source.name] = shine_through_loads(
        beam,
        deposition,
        wall,
        divergence=0.004,
        resample=5,
        seed=k,
        map_bins=(48, 240),
    )
    loads = armour[source.name]
    print(
        f"{source.name}: shine-through {deposition.shine_through_fraction:.1%} of "
        f"{deposition.injected_power / 1e6:.1f} MW, on the wall {loads.attrs['wall_power'] / 1e3:.0f} kW, "
        f"missed {loads.attrs['missed_power'] / 1e3:.1f} kW, peak {loads.attrs['peak_wall_load'] / 1e6:.2f} MW/m^2"
    )
print(f"{time.perf_counter() - start:.0f} s")

In [ ]:
for name, loads in armour.items():
    heat = loads.wall_load_map.fillna(0.0) / 1e6
    hottest = heat.argmax(...)
    theta0, phi0 = (float(heat[c][hottest[c]]) for c in ("theta", "phi"))
    window = heat.sel(
        theta=slice(theta0 - 1.2, theta0 + 1.2), phi=slice(phi0 - 0.3, phi0 + 0.3)
    )
    window.attrs.update(long_name="heat flux", units="MW/m^2")
    window.plasma.plot.view(x="phi", y="theta", title=name).slice().show()

At this density each source shines about 13 % of its 1.8 MW through, some 230 kW onto the stand-in wall. The two footprints lie on the inboard side, $\theta \approx 3.7$ and $\varphi \approx 0.75$ to $0.8$ rad, shifted by a few hundredths of a radian in $\varphi$ against each other, so they would share one armour panel. On bins of about 6 cm by 13 cm the peak is about 0.9 MW/m²; the face peaks printed above (1.2 to 1.5 MW/m²) are dominated by counting noise, with a few dozen lines on the hottest faces of this fine triangulation, and need more markers before they are read as numbers. The 0.5 kW missed by source 1 are lines at the edge of its divergence that pass beside the stand-in wall without entering it; on a real machine they would hit the port.

## What remains

- The armour: a real first wall (IPP's CAD for W7-X, an STL read with `TriangleMesh.read`) with its beam-dump tiles, where the footprint would be checked against the tiles' design load.
- The re-ionisation of the beam in the neutral gas between the plasma and the wall, and the reflection of fast atoms from the wall, are left out: all the surviving power is deposited where the line hits.
- A comparison with BBNBI/ASCOT5's shine-through wall loads on the same wall and beam.